# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict
print(f"{metadata.name}: {metadata.description}")

### Dataset Metadata

- **Title**: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
- **Identifier**: 10.71728/senscience.qs2f-h81p
- **RecordSet**: See details below
- **Description**: Tabular dataset of 77 cancer survivors with second primary colorectal cancer, including clinical and pathological variables such as demographics, comorbidities, first and second primary cancer types, treatment history, intervals between diagnoses, anatomical location of colorectal cancer, histopathological subtype, presence of distant metastasis, and microsatellite instability status. Data supports investigation of clinicopathological predictors and distribution of MSI-H phenotype.

## 2. Data Overview

Review available record sets, fields, and their `@id` values.

Croissant datasets define data structure using *record sets*, where each record set has fields, and each field contains columns. Each data element can be referenced by its unique `@id`.

Let's list all `recordSet` `@id`s and get an overview of their fields and columns.

In [ ]:
# List all record sets and their fields (@id for everything)
record_sets_info = []
for record_set in dataset.record_sets:
    print(f"RecordSet name: {record_set.name if hasattr(record_set, 'name') else ''}\n  @id: {record_set.id}")
    record_sets_info.append({'name': getattr(record_set, 'name', ''), 'id': record_set.id})
    print("  Fields:")
    for field in record_set.fields:
        col_names = [col.name for col in getattr(field, 'columns', [])]
        print(f"    Field: {field.name if hasattr(field, 'name') else ''}, @id: {field.id}, Columns: {col_names}")
    print('-'*40)
if not record_sets_info:
    print("No record sets found directly in dataset.metadata, fetching from dataset.record_sets API...")
    # If .recordSets is empty, .record_sets is an API of mlcroissant
    # The above loop should have covered it.
else:
    print(f"Found {len(record_sets_info)} record sets.")

## 3. Data Extraction

Now, we can load records from a specific record set using its `@id`. Examine the `@id` for the record set of most interest (e.g., for the main patient or observation table) and load it into a DataFrame.

We'll extract data for all major record sets. Please revisit the cell above and copy out the `@id` values—you must reference entities always by their `@id`, not just their name.

In [ ]:
# Prepare to extract data for all available record sets
record_sets_ids = [rs['id'] for rs in record_sets_info]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Extracting records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# Preview the structure of the first available DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"DataFrame columns for RecordSet @id: {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No tabular data found for any record set.")

## 4. Exploratory Data Analysis (EDA)

In this section, we'll process the main record set. You may need to manually adjust the `record_set_id` below to match the patient or case-level table identified earlier.

We'll do the following:
- Select a numeric field (referenced by `@id`)
- Filter records with a threshold
- Normalize the chosen numeric field
- Group by a categorical field (also using `@id`, e.g., anatomical location)

You must know your selected field's `@id` and column label from the earlier output.

In [ ]:
# Choose which RecordSet to analyze (update these @id values to fit your dataset):
main_record_set_id = None
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]

    print(f"Analyzing RecordSet @id: {main_record_set_id}")
    print("DataFrame columns:", df.columns.tolist())

    # Choose a numeric field, use @id from earlier output, plug into column name
    # Example: suppose the field '@id' is 'cr:Age' and column in dataframe is 'Age'
    numeric_field = None
    candidate_numeric_cols = [col for col in df.columns if df[col].dtype.kind in 'if']
    print("Numeric candidate fields:", candidate_numeric_cols)
    if candidate_numeric_cols:
        numeric_field = candidate_numeric_cols[0]

    # Use threshold for filtering
    if numeric_field is not None:
        threshold = df[numeric_field].quantile(0.25)  # example: lower quartile
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records: {filtered_df.shape[0]} rows where {numeric_field} > {threshold:.2f}")

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No numeric field found to analyze.")

    # Choose a group/categorical field (e.g., anatomical_location or status)
    group_field = None
    candidate_group_cols = [col for col in df.columns if df[col].dtype == 'object']
    print("Categorical candidate fields:", candidate_group_cols)
    if candidate_group_cols:
        group_field = candidate_group_cols[0]

    if numeric_field and group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"Mean {numeric_field} per {group_field} (top 10):\n", grouped_df.head(10))
else:
    print("No main record set DataFrame available for EDA.")

## 5. Visualization

Visualize the distribution of the numeric field and the mean per group after processing. This section assumes previous code selected a numeric and a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and main_record_set_id is not None and numeric_field and group_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=filtered_df, x=numeric_field, bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    plt.figure(figsize=(10, 5))
    # Group and sort
    group_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    sns.barplot(y=group_means.index, x=group_means.values, orient='h')
    plt.xlabel(f'Mean {numeric_field}')
    plt.ylabel(group_field)
    plt.title(f"Mean {numeric_field} per {group_field}")
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot: fields or DataFrame unavailable.")

## 6. Conclusion

- We loaded the FAIR^2 colorectal cancer survivor dataset using `mlcroissant`.
- All data structures and fields were referenced by their `@id` values as required by the Croissant schema.
- Explored and visualized numeric and categorical patterns in the clinical data table (using field `@id`s and DataFrame column names).
- This structured workflow allows reproducible and standards-based data science on FAIR datasets using only metadata and schema URLs.

**Next steps**: Consider integrating advanced analyses or linking Croissant `@id` fields in modeling workflows.